In [140]:
import math
import random
import numpy
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
batch_size = 8
block_size = 8
n_embd = 16
n_head = 4
epochs = 30000
learning_rate = 1e-3
n_layers = 4
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"using device: {device}")

In [142]:
with open('data/grabbed_nietzsche.txt', 'r', encoding='utf-8') as f:
    nietzshe = f.read()

In [143]:
len(nietzshe)

1772469

In [144]:
characters = sorted(set(nietzshe))
vocab_size = len(characters)
print(''.join(characters))
print(vocab_size)


 !"&'()*,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]_abcdefghijklmnopqrstuvwxyz}§ÆàâäæèéêîïôöüœάέήίαβγδεζηθικλμνξοπρςστυφχωόώἀἄἆἈἐἑἒἰὀὖ᾽ῑ–—‘’“”
148


In [145]:
stoi = {s:i for i, s in enumerate(characters)}
itos = {i:s for s, i in stoi.items()}
print(stoi)
print(itos)                                     # the encoders we are using are character to token encoders

encode = lambda s: [stoi[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])
encode('hi there')
decode(encode("hi there"))


{'\n': 0, ' ': 1, '!': 2, '"': 3, '&': 4, "'": 5, '(': 6, ')': 7, '*': 8, ',': 9, '-': 10, '.': 11, '/': 12, '0': 13, '1': 14, '2': 15, '3': 16, '4': 17, '5': 18, '6': 19, '7': 20, '8': 21, '9': 22, ':': 23, ';': 24, '<': 25, '=': 26, '>': 27, '?': 28, 'A': 29, 'B': 30, 'C': 31, 'D': 32, 'E': 33, 'F': 34, 'G': 35, 'H': 36, 'I': 37, 'J': 38, 'K': 39, 'L': 40, 'M': 41, 'N': 42, 'O': 43, 'P': 44, 'Q': 45, 'R': 46, 'S': 47, 'T': 48, 'U': 49, 'V': 50, 'W': 51, 'X': 52, 'Y': 53, 'Z': 54, '[': 55, ']': 56, '_': 57, 'a': 58, 'b': 59, 'c': 60, 'd': 61, 'e': 62, 'f': 63, 'g': 64, 'h': 65, 'i': 66, 'j': 67, 'k': 68, 'l': 69, 'm': 70, 'n': 71, 'o': 72, 'p': 73, 'q': 74, 'r': 75, 's': 76, 't': 77, 'u': 78, 'v': 79, 'w': 80, 'x': 81, 'y': 82, 'z': 83, '}': 84, '§': 85, 'Æ': 86, 'à': 87, 'â': 88, 'ä': 89, 'æ': 90, 'è': 91, 'é': 92, 'ê': 93, 'î': 94, 'ï': 95, 'ô': 96, 'ö': 97, 'ü': 98, 'œ': 99, 'ά': 100, 'έ': 101, 'ή': 102, 'ί': 103, 'α': 104, 'β': 105, 'γ': 106, 'δ': 107, 'ε': 108, 'ζ': 109, 'η': 110

'hi there'

In [146]:
tokenized_nietzshe = torch.tensor(encode(nietzshe), dtype=torch.long)       # encode the entirety of nietzshe into tokens
                                                                            

tokenized_nietzshe[:1000]

tensor([ 48,  36,  33,   1,  48,  51,  37,  40,  37,  35,  36,  48,   1,  43,
         34,   1,  48,  36,  33,   1,  37,  32,  43,  40,  47,   0,   0,  30,
         53,   0,   0,  34,  46,  37,  33,  32,  46,  37,  31,  36,   1,  42,
         37,  33,  48,  54,  47,  31,  36,  33,   0,   0,  43,  75,   9,   1,
         36,  72,  80,   1,  77,  72,   1,  44,  65,  66,  69,  72,  76,  72,
         73,  65,  66,  76,  62,   1,  80,  66,  77,  65,   1,  77,  65,  62,
          1,  36,  58,  70,  70,  62,  75,   0,   0,   0,  48,  36,  33,   1,
         29,  42,  48,  37,  31,  36,  46,  37,  47,  48,   0,   0,  57,  42,
         43,  48,  33,  47,   1,  48,  43,   1,  54,  29,  46,  29,  48,  36,
         49,  47,  48,  46,  29,   9,   1,  29,  42,  32,   1,  33,  48,  33,
         46,  42,  29,  40,   1,  46,  33,  31,  49,  46,  46,  33,  42,  31,
         33,  57,   0,   0,   0,  48,  46,  29,  42,  47,  40,  29,  48,  33,
         32,   1,  30,  53,   0,   0,  29,  42,  48,  36,  43,  

In [147]:
# split into train and test data
n = int(len(tokenized_nietzshe)*.9)
train_nietzshe = tokenized_nietzshe[:n]
test_nietzshe = tokenized_nietzshe[n:]

print(train_nietzshe.shape, test_nietzshe.shape)

torch.Size([1595222]) torch.Size([177247])


In [148]:
train_nietzshe[:block_size+1]

tensor([48, 36, 33,  1, 48, 51, 37, 40, 37])

In [149]:
v = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset
for i in v:
    print(i.item())

530856
838448
195446
724537
533761
788728
1531955
696701


In [ ]:

def generate_batch(type=None):
    xb = []
    yb = []

    if type == 'test':
        data = test_nietzshe
    else:
        data = train_nietzshe

    n = len(data)
    start_vals = torch.randint(n-block_size, (batch_size, ))     # how to sample random values from the trainset 
    for start_val in start_vals:
        start_val = start_val.item()
        xb.append(data[start_val:start_val+block_size])
        yb.append(data[start_val+1:start_val+block_size+1])
    
    xb = torch.stack(xb)
    yb = torch.stack(yb)
    return xb.to(device), yb.to(device)                          # move batches onto the active device

xb, yb = generate_batch()

print(xb, yb)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size)
        self.query = nn.Linear(n_embd, head_size)
        self.value = nn.Linear(n_embd, head_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, xb):                                              # (B, T, C)
        k = self.key(xb)                                                # (B, T, head_size=n_embd/n_head)
        q = self.query(xb)                                              # (B, T, head_size)
        v = self.value(xb)                                              # (B, T, head_size)

        B, T, C = k.shape

        weights = q @ k.transpose(-2, -1) / C**0.5                      # (B, T, C) @ (B, C, T) => (B, T, T)
                                                                        # divided by n_embd root to normalize the values, bring them to one
        mask = torch.tril(torch.ones(T, T, device=xb.device))          # triangular mask, built on the same device as the input
        weights = weights.masked_fill(mask == 0, float('-inf'))      
        weights = F.softmax(weights, dim=-1)                            # softmax to get probabilities (B, T, T)
        weights = self.dropout(weights)

        out = weights @ v                                               # (B, T, T) @ (B, T, head_size) => (B, T, head_size)
        return out
      

In [152]:
class MultipleSelfAttention(nn.Module):
    def __init__(self, n_head, n_embd):
        super().__init__()
        self.heads = nn.ModuleList([SelfAttention(n_embd // n_head) for i in range(n_head)])
        self.lin_proj = nn.Linear(n_embd, n_embd)

    def forward(self, xb):
        full_attention = []

        for head in self.heads: 
            full_attention.append(head(xb))                             # (B, T, head_size)
        
        full_attention = torch.cat(full_attention, dim=-1)              # add n_head of (B, T, head_size) along last dimension = (B, T, C)
        full_attention = self.lin_proj(full_attention)                  # linear projection of (B, T, C) to get (B, T, C)

        return full_attention


In [153]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.GELU(),
            nn.Linear(4*n_embd, n_embd),
            nn.Dropout(dropout)
        )
    
    def forward(self, xb):
        return self.ff(xb)

In [154]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.full_attend = MultipleSelfAttention(n_head, n_embd)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, xb):
        attention = xb + self.full_attend(self.ln1(xb))                      # (B, T, C) => 4 * self_attend(B, T, C/4) => (B, T, C)
        feed_forward = attention + self.ff(self.ln2(attention))

        return feed_forward

In [ ]:
class LTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)         # a token embedding allows us to represent letters as vectors
        self.pos_embedding = nn.Embedding(block_size, n_embd)           # attention also depends on the location of a token
                                                                        # the token embedding itself will give the same value to a character at the front vs at the end
                                                                        # pos embedding handles this
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for i in range(n_layers)])
        self.last_layer = nn.Linear(n_embd, vocab_size)

    def forward(self, xb, targets):
        B, T = xb.shape

        tok_embd = self.token_embedding(xb)                             # (B, T, C)
        pos_embd = self.pos_embedding(torch.arange(T, device=xb.device))  # (T, C), positions built on the input's device
        x = tok_embd + pos_embd                                       # (B, T, C) + (T, C) => (B, T, C) + (B, T, C)

        out = self.blocks(x)
        logits = self.last_layer(out)
        
        if targets == None: 
            loss = None
        else:
            flat_logits = logits.view(-1, vocab_size)
            flat_targets = targets.view(-1)
            loss = F.cross_entropy(flat_logits, flat_targets)
        return logits, loss
    
    def generate(self, tokens, idx):

        for i in range(tokens):
            logits, loss = self(idx[:, -block_size:], None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_tok), dim=1)
        
        return idx


xb, yb = generate_batch()

model = LTransformer().to(device)                                   # move all model parameters onto the active device
logits, loss = model(xb, yb)
print(logits, loss)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

def estimate_loss(iters):
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(iters)
        for k in range(iters):
            xb, yb = generate_batch(split)                  # use the split's data (train vs test)
            tlogits, tloss = model(xb, yb)
            losses[k] = tloss.item()
        out[split] = losses.mean()
    model.train()
    return out

for i in range(epochs):
    xb, yb = generate_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if i % 1000 ==0:
        estimated_loss = estimate_loss(20)
        print(f"loss at epoch {i}: {estimated_loss['train']:.4f}, {estimated_loss['test']:.4f}")

In [ ]:
idx = torch.zeros((1, 1), dtype=torch.long, device=device)          # seed token, on the active device
out = model.generate(100, idx)
print(decode(out[0].tolist()))